In [ ]:
from huggingface_hub import HfApi
import pandas as pd
import time


In [ ]:
#API token
HF_TOKEN = "X"
api = HfApi(token=HF_TOKEN)

In [ ]:
#Setting the number of models, licenses and keywords
TOP_N_MODELS = 10000

OPEN_SOURCE_LICENSES = {
    "apache-2.0",
    "mit",
    "bsd-3-clause",
    "gpl-3.0",
    "agpl-3.0",
    "cc-by-4.0",
    "cc-by-sa-4.0",
}

SECURITY_KEYWORDS = [
    "security", "vulnerability", "vulnerabilities", "adversarial",
    "attack", "attacks", "poison", "poisoning",
    "risk", "risks", "mitigation", "robust",
    "eval", "evaluation", "limitations", "safety"
]


In [ ]:
# Retrieves the model's licence information by checking multiple metadata
# fields. Hugging Face stores licence declarations inconsistently, so this
# function first checks the config and then falls back to cardData. Returns
# the licence string if found, otherwise None.

def extract_license(model_info):
    if model_info.config and "license" in model_info.config:
        return str(model_info.config["license"]).lower()

    if model_info.cardData:
        lic = model_info.cardData.get("license")
        if lic:
            return str(lic).lower()

    return None


def get_readme(model_id):
    try:
        info = api.model_info(model_id)
        if info.cardData and "text" in info.cardData:
            return str(info.cardData["text"])
        else:
            return str(info.cardData or "")
    except:
        return ""


def find_security_keywords(text):
    text = text.lower()
    return [kw for kw in SECURITY_KEYWORDS if kw in text]

In [13]:
# Retrieves the top downloaded models from Hugging Face, filters them by
# open-source licence and documentation content, and identifies models that
# mention security-related terms in their README/model card. Returns a DataFrame
# containing only the models that match the security criteria

def scrape_top_500_security_models():
    all_rows = []

    print(f"Fetching Top {TOP_N_MODELS} downloaded Hugging Face models...")
    models = api.list_models(
        sort="downloads",
        direction=-1,
        limit=TOP_N_MODELS,
        full=True
    )

    for model in models:

        # Fetch model info
        try:
            info = api.model_info(model.modelId)
        except:
            continue

        # License filtering
        license_id = extract_license(info)
        if not license_id or license_id not in OPEN_SOURCE_LICENSES:
            continue

        # Extract README
        readme = get_readme(model.modelId)

        # Scan for security keywords
        matches = find_security_keywords(readme)
        if matches:
            print("FOUND:", model.modelId)

            all_rows.append({
                "modelId": model.modelId,
                "downloads": model.downloads,
                "likes": model.likes,
                "license": license_id,
                "tags": model.tags,
                "keywords_found": matches,
                "readme_length": len(readme),
                "url": f"https://huggingface.co/{model.modelId}",
            })

        time.sleep(0.3)

    return pd.DataFrame(all_rows)





Fetching TOP 10000 downloaded Hugging Face models...
FOUND: jonatasgrosman/wav2vec2-large-xlsr-53-russian
FOUND: jonatasgrosman/wav2vec2-large-xlsr-53-portuguese
FOUND: BAAI/bge-large-en-v1.5
FOUND: cardiffnlp/twitter-roberta-base-sentiment-latest
FOUND: jonatasgrosman/wav2vec2-large-xlsr-53-dutch
FOUND: BAAI/bge-base-en-v1.5
FOUND: amazon/chronos-2
FOUND: BAAI/bge-small-en-v1.5
FOUND: Alibaba-NLP/gte-large-en-v1.5
FOUND: intfloat/multilingual-e5-large
FOUND: jonatasgrosman/wav2vec2-large-xlsr-53-polish
FOUND: opensearch-project/opensearch-neural-sparse-encoding-doc-v2-distill
FOUND: mpoyraz/wav2vec2-xls-r-300m-cv7-turkish
FOUND: intfloat/multilingual-e5-small
FOUND: mixedbread-ai/mxbai-embed-large-v1
FOUND: intfloat/multilingual-e5-base
FOUND: nomic-ai/nomic-embed-text-v1.5
FOUND: Snowflake/snowflake-arctic-embed-l-v2.0
FOUND: vidore/colqwen2.5-v0.2
FOUND: intfloat/e5-base-v2
FOUND: Alibaba-NLP/gte-multilingual-base
FOUND: WhereIsAI/UAE-Large-V1
FOUND: intfloat/multilingual-e5-large-i

Invalid model-index. Not loading eval results into CardData.


FOUND: gigant/romanian-wav2vec2
FOUND: intfloat/e5-large-v2
FOUND: thenlper/gte-large
FOUND: minishlab/potion-base-32M
FOUND: OpenMed/OpenMed-NER-PharmaDetect-BigMed-278M
FOUND: dunzhang/stella-mrl-large-zh-v3.5-1792d
FOUND: OpenMed/OpenMed-NER-GenomicDetect-PubMed-109M
FOUND: deepset/roberta-base-squad2
FOUND: nomic-ai/nomic-embed-text-v1


Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.


FOUND: thenlper/gte-small
FOUND: airesearch/wav2vec2-large-xlsr-53-th
FOUND: comodoro/wav2vec2-xls-r-300m-cs-250
FOUND: OpenMed/OpenMed-NER-PharmaDetect-ModernClinical-149M
FOUND: kingabzpro/wav2vec2-large-xls-r-300m-Urdu
FOUND: OpenMed/OpenMed-NER-BloodCancerDetect-TinyMed-65M
FOUND: OpenMed/OpenMed-NER-GenomicDetect-BigMed-560M
FOUND: Snowflake/snowflake-arctic-embed-m
FOUND: OpenMed/OpenMed-NER-PharmaDetect-BioPatient-108M
FOUND: OpenMed/OpenMed-NER-BloodCancerDetect-ElectraMed-560M
FOUND: hkunlp/instructor-xl
FOUND: OpenMed/OpenMed-NER-GenomicDetect-PubMed-335M
FOUND: intfloat/e5-large


...

FOUND: bartowski/TheBeagle-v2beta-32B-MGS-GGUF
FOUND: ytu-ce-cosmos/turkish-gpt2-large-750m-instruct-v0.1
FOUND: lmms-lab/llava-onevision-qwen2-0.5b-si
FOUND: SanctumAI/granite-8b-code-instruct-GGUF
FOUND: lmstudio-community/deepseek-coder-6.7B-kexer-GGUF
FOUND: ApsaraStackMaaS/EvoQwen2.5-VL-Retriever-3B-v1
FOUND: lmstudio-community/dolphin-2.8-mistral-7b-v02-GGUF
FOUND: dnhkng/RYS-Phi-3-medium-4k-instruct
FOUND: BEE-spoke-data/smol_llama-220M-GQA
FOUND: lmms-lab/llava-onevision-qwen2-7b-si


Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.


FOUND: sayed0am/arabic-english-bge-m3
FOUND: ggml-org/bge-small-en-v1.5-Q8_0-GGUF
FOUND: geoffmunn/Qwen3Guard-Stream-0.6B
FOUND: dwzhu/e5-base-4k


Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.


FOUND: Sunbird/Sunflower-32B

Saved: 218 models → huggingface_security_top500_scan.csv


In [ ]:
#Saving the results in a CSV format
df = scrape_top_500_security_models()
df.to_csv("huggingface_security_top500_scan.csv", index=False)

print(f"\nSaved: {len(df)} models → huggingface_security_top500_scan.csv")

In [14]:
from google.colab import files
files.download("huggingface_security_top500_scan.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>